# Pipeline de unificación de datos: Isótopos, Rasgos funcionales y Dinámica forestal

Este cuaderno une tres bases de datos en una sola tabla, para poder analizarlas juntas y detectar
qué información falta.

**Las tres bases:**

1. **Isótopos** (`isotopos_concatenado.xlsx`) → es la base **más importante / de referencia**.
2. **Rasgos funcionales** (`Rasgos_unificado_todas_las_bases.xlsx`) → se une a la base de isótopos.
3. **Dinámica forestal** (`DinamicaForestal_unificada_recensos.csv`) → de aquí solo tomamos el **DBH**
   (diámetro a la altura del pecho) y lo agregamos al resultado anterior.

**Reglas del pipeline** (definidas por el usuario, incluyendo la segunda ronda de correcciones):

- La unión entre Isótopos y Rasgos se hace con un **outer join** (unión externa), para ver claramente
  qué filas de una base no tienen pareja en la otra.
- Los **rasgos** solo existen para los años 2023, 2024 y 2025. Los **isótopos** tienen muestras de
  varios años (2005, 2006, 2007, 2009, 2011, 2023, 2025).
- **[CORRECCIÓN]** El **año de la base de rasgos manda** (es el año "real" de muestreo); el año de la
  base de isótopos es **referencial**. Por ejemplo, en Galeras hay rasgos etiquetados como 2024 y
  2025, pero sus isótopos están todos etiquetados como 2025 — son el mismo muestreo de campo, solo que
  el laboratorio de isótopos registró el año de procesamiento, no el de colecta. Por eso, dentro del
  período reciente (2023-2025), la unión con isótopos ya **no exige que el año coincida exactamente**;
  solo exige que coincida el árbol. Los isótopos **antiguos** (2006 y anteriores) se mantienen
  totalmente separados de este emparejamiento, para no mezclarlos por error con los rasgos actuales.
- **[CORRECCIÓN]** Se corrigió un error en el que las filas de isótopos históricos (2006, etc.)
  aparecían con el `Site` (y otras columnas propias de isótopos) vacío en la tabla final. La causa era
  un desajuste de nombres de columnas al combinar el bloque "reciente" con el "histórico" — ya
  corregido (ver Paso 4).
- **[NUEVO]** Se agrega el cálculo de **área basal** por árbol y por censo (a partir del DBH), y el
  **cambio (delta) en el tiempo** tanto del área basal (crecimiento) como de los isótopos (δ13C, δ15N,
  [C]%, [N]). El área basal se calcula en el mismo paso que el DBH (misma granularidad, cálculo
  directo). Los deltas, en cambio, son un tipo de resultado distinto — pasan de "una fila por muestra"
  a "una fila por árbol resumida en el tiempo" — así que se calculan en un bloque propio dentro de
  este mismo notebook y se entregan tanto como columnas repetidas en la tabla larga principal, como en
  dos hojas de resumen dedicadas (una fila por árbol), para que sea fácil seguir analizándolos después
  sin tener que reconstruir el cálculo desde cero.
- **[CORRECCIÓN]** Se **excluye la base "SF"** (dataset_source = `SF`, sitios CaNu/BoN/SFNu) de la
  unión, porque el usuario detectó problemas de calidad en esa base.
- Siempre que sea posible se usa el **new tree ID** (el ID "nuevo" de 2025) como identificador
  preferido para unir las tablas.
- **[CORRECCIÓN]** El **DBH ahora se toma del censo disponible más reciente** para cada árbol
  (censo 3 → si no existe, censo 2 → si no existe, censo 1), en vez de usar solo el censo 3. Así el
  DBH de un árbol se "repite" correctamente en todas sus filas (por ejemplo, si el árbol aparece tanto
  en una fila de isótopos de 2006 como en una de 2025, el mismo DBH aparece en ambas), incluso si ese
  árbol solo fue medido en el censo 1.

El código está escrito con un nivel básico/intermedio de Python y pandas, con comentarios en cada paso.


## Paso 0 — Importar librerías

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## Paso 1 — Cargar las tres bases de datos

Ajusta `DATA_DIR` si tus archivos están en otra carpeta.


In [2]:
DATA_DIR = "data"

iso = pd.read_excel(f"{DATA_DIR}/isotopos_concatenado.xlsx", sheet_name="Datos_concatenados")
rasgos = pd.read_excel(f"{DATA_DIR}/Rasgos_unificado_todas_las_bases.xlsx", sheet_name="Sheet1")
dinamica = pd.read_csv(f"{DATA_DIR}/DinamicaForestal_unificada_recensos.csv")

print("Isótopos:        ", iso.shape, "(filas, columnas)")
print("Rasgos:           ", rasgos.shape, "(filas, columnas)")
print("Dinámica forestal:", dinamica.shape, "(filas, columnas)")


Isótopos:         (669, 27) (filas, columnas)
Rasgos:            (313, 36) (filas, columnas)
Dinámica forestal: (1336, 25) (filas, columnas)


## Paso 2 — Excluir la base "SF" de rasgos

El usuario identificó problemas de calidad en la base `SF` (sitios `CaNu`, `BoN`, `SFNu`, cargados
desde `data_for_R.xlsx`). La excluimos **antes** de cualquier unión, para que no contamine el
resultado. (La base de isótopos no tiene esos sitios, así que no hace falta tocarla).


In [3]:
print("Filas por dataset_source en rasgos (antes de excluir):")
print(rasgos["dataset_source"].value_counts())

n_antes = len(rasgos)
rasgos = rasgos[rasgos["dataset_source"] != "SF"].copy()
n_despues = len(rasgos)

print(f"\nSe excluyeron {n_antes - n_despues} filas de la base 'SF'.")
print(f"Quedan {n_despues} filas en rasgos.")


Filas por dataset_source en rasgos (antes de excluir):
dataset_source
Galeras                90
SelvaViva              62
Oyacachi_Guacamayos    61
SF                     51
Sumaco_2025            49
Name: count, dtype: int64

Se excluyeron 51 filas de la base 'SF'.
Quedan 262 filas en rasgos.


## Paso 3 — Crear un identificador de árbol único (`id_arbol`) en cada base

Cada base identifica a los árboles de forma un poco distinta:

| Base      | Columna con ID "nuevo" (2025) | Columna con ID "viejo" |
|-----------|-------------------------------|--------------------------|
| Isótopos  | `new_TreeID`                  | `old_treeID`             |
| Rasgos    | `new_tree_ID_2025`             | `treeID`                 |
| Dinámica  | `new_tree_ID`                  | `treeID`                 |

**Estrategia** (usando el *new tree ID* siempre que sea posible):

1. Convertimos ambas columnas de ID a número.
2. Creamos `id_arbol`: usa el **ID nuevo** si existe; si no, usa el **ID viejo** como respaldo.

También guardamos el año de cada base por separado, con nombres distintos para no confundirlos:
`anio_rasgos` (de la fecha de muestreo) y `anio_isotopo` (de la columna `Year` de isótopos).


In [4]:
def a_numero(serie):
    '''Convierte una columna a número; lo que no se pueda convertir queda como NaN (vacío).'''
    return pd.to_numeric(serie, errors="coerce")


# --- Isótopos ---
iso["id_nuevo"] = a_numero(iso["new_TreeID"])
iso["id_viejo"] = a_numero(iso["old_treeID"])
iso["id_arbol"] = iso["id_nuevo"].combine_first(iso["id_viejo"])
iso["anio_isotopo"] = a_numero(iso["Year"])

# --- Rasgos ---
rasgos["id_nuevo"] = a_numero(rasgos["new_tree_ID_2025"])
rasgos["id_viejo"] = a_numero(rasgos["treeID"])
rasgos["id_arbol"] = rasgos["id_nuevo"].combine_first(rasgos["id_viejo"])
rasgos["anio_rasgos"] = pd.to_datetime(rasgos["sampling_date"], errors="coerce").dt.year

# --- Dinámica forestal ---
dinamica["id_nuevo"] = a_numero(dinamica["new_tree_ID"])
dinamica["id_viejo"] = a_numero(dinamica["treeID"])
dinamica["id_arbol"] = dinamica["id_nuevo"].combine_first(dinamica["id_viejo"])

print("Filas SIN id_arbol (ni nuevo ni viejo disponible):")
print("  Isótopos: ", iso["id_arbol"].isna().sum(), "de", len(iso))
print("  Rasgos:   ", rasgos["id_arbol"].isna().sum(), "de", len(rasgos))
print("  Dinámica: ", dinamica["id_arbol"].isna().sum(), "de", len(dinamica))
print()
print("Filas de RASGOS sin año de muestreo (sampling_date vacío):", rasgos["anio_rasgos"].isna().sum())


Filas SIN id_arbol (ni nuevo ni viejo disponible):
  Isótopos:  0 de 669
  Rasgos:    0 de 262
  Dinámica:  82 de 1336

Filas de RASGOS sin año de muestreo (sampling_date vacío): 12


## Paso 4 — Unión 1: Isótopos ⟷ Rasgos

### La corrección clave de esta versión

Antes exigíamos que coincidiera el **ID del árbol y el año exacto**. Eso causaba un problema real: en
Galeras, el mismo muestreo de campo quedó etiquetado como **2024** en la base de rasgos, pero como
**2025** en la base de isótopos (probablemente por la fecha de procesamiento en laboratorio vs. la
fecha de colecta en campo). Como los años no coincidían exactamente, esas filas no se unían — aunque
en realidad sí eran el mismo árbol y el mismo muestreo.

**Solución:** dividimos los isótopos en dos grupos:

- **Isótopos recientes** (2023, 2024 o 2025): se unen con rasgos usando **solo el `id_arbol`**, sin
  exigir que el año coincida exactamente. El año que se conserva en la tabla final es el de la base de
  **rasgos** (más confiable), y se guarda el año de isótopos aparte (`anio_isotopo`) solo como
  referencia.
- **Isótopos antiguos** (2006 y anteriores): se mantienen **separados**, sin intentar emparejarlos con
  rasgos. Esto evita mezclar por error un isótopo de hace 20 años con un rasgo actual del mismo árbol.
  (Verificamos que ningún árbol tiene isótopos en más de un año reciente a la vez, así que unir solo
  por `id_arbol` dentro de 2023-2025 no genera parejas ambiguas).


In [5]:
ANIOS_RASGOS = [2023, 2024, 2025]

# Verificación: ¿algún árbol tiene isótopos en MÁS de un año reciente? (si los hubiera, unir solo por
# id_arbol podría generar una unión ambigua con más de un candidato del mismo período)
recientes_chequeo = iso[iso["anio_isotopo"].isin(ANIOS_RASGOS)]
anios_por_arbol = recientes_chequeo.groupby("id_arbol")["anio_isotopo"].apply(lambda s: sorted(set(s.dropna())))
arboles_ambiguos = anios_por_arbol[anios_por_arbol.apply(len) > 1]
print("Árboles con isótopos en más de un año reciente (2023-2025):", len(arboles_ambiguos))
print("(si este número es 0, es seguro unir por id_arbol sin exigir año exacto)")


Árboles con isótopos en más de un año reciente (2023-2025): 0
(si este número es 0, es seguro unir por id_arbol sin exigir año exacto)


In [6]:
# Separamos los isótopos en "recientes" (2023-2025) y "históricos" (todo lo demás, ej. 2006)
iso_reciente = iso[iso["anio_isotopo"].isin(ANIOS_RASGOS)].copy()
iso_historico = iso[~iso["anio_isotopo"].isin(ANIOS_RASGOS)].copy()

print("Isótopos recientes (2023-2025):", len(iso_reciente), "filas")
print("Isótopos históricos (2006 y otros años previos):", len(iso_historico), "filas")


Isótopos recientes (2023-2025): 258 filas
Isótopos históricos (2006 y otros años previos): 411 filas


In [7]:
# Unión de isótopos recientes + rasgos, SOLO por id_arbol (sin exigir año exacto)
merge_reciente = pd.merge(
    iso_reciente, rasgos,
    on="id_arbol",
    how="outer",
    suffixes=("_iso", "_rasgos"),
)

# El año final prioriza el de RASGOS (más confiable); si esa fila no tiene rasgo, usa el de isótopos
merge_reciente["anio"] = merge_reciente["anio_rasgos"].combine_first(merge_reciente["anio_isotopo"])

merge_reciente["tiene_isotopos"] = merge_reciente["Sample_ID"].notna()
merge_reciente["tiene_rasgos"] = merge_reciente["PlotID"].notna()

print("Filas resultantes (solo período reciente):", merge_reciente.shape[0])
print("Coincidencias isótopos + rasgos (mismo árbol, dentro de 2023-2025):",
      (merge_reciente["tiene_isotopos"] & merge_reciente["tiene_rasgos"]).sum())


Filas resultantes (solo período reciente): 283
Coincidencias isótopos + rasgos (mismo árbol, dentro de 2023-2025): 258


In [8]:
# IMPORTANTE: algunas columnas existen en AMBAS bases (Site, Plot, family, genus, species, QC_flag,
# comment, source_file). En "merge_reciente" esas columnas ya quedaron renombradas con el sufijo
# "_iso" o "_rasgos" (por el parámetro suffixes del merge). Pero "iso_historico" nunca pasó por ese
# merge, así que esas columnas todavía se llaman "Site", "Plot", etc. (sin sufijo).
#
# Si concatenáramos así nada más, pandas alinearía por NOMBRE de columna: como "Site" no es lo mismo
# que "Site_iso", terminaríamos con una columna "Site_iso" vacía para las filas históricas y una
# columna "Site" aparte solo para ellas — perdiendo esa información en la columna que sí se usa en el
# resto de la tabla. Por eso, primero renombramos esas columnas en "iso_historico" para que coincidan
# con el mismo esquema "_iso" que ya tiene "merge_reciente".
columnas_comunes = (set(iso.columns) & set(rasgos.columns)) - {"id_arbol"}
mapa_renombrado = {col: f"{col}_iso" for col in columnas_comunes if col in iso_historico.columns}
iso_historico = iso_historico.rename(columns=mapa_renombrado)

# Los isótopos históricos se agregan tal cual, sin pareja de rasgos (las columnas de rasgos
# quedarán vacías para estas filas — es lo esperado hasta que el usuario incorpore esos datos)
iso_historico["anio"] = iso_historico["anio_isotopo"]
iso_historico["tiene_isotopos"] = True
iso_historico["tiene_rasgos"] = False

# Unimos ambos bloques en una sola tabla
merge1 = pd.concat([merge_reciente, iso_historico], ignore_index=True, sort=False)

print("Filas totales de la unión Isótopos + Rasgos:", merge1.shape[0])
print()
n_ambos = (merge1["tiene_isotopos"] & merge1["tiene_rasgos"]).sum()
n_solo_iso = (merge1["tiene_isotopos"] & ~merge1["tiene_rasgos"]).sum()
n_solo_rasgos = (~merge1["tiene_isotopos"] & merge1["tiene_rasgos"]).sum()
print(f"Filas con isótopos Y rasgos:      {n_ambos}")
print(f"Filas SOLO con isótopos:          {n_solo_iso}")
print(f"Filas SOLO con rasgos:            {n_solo_rasgos}")


Filas totales de la unión Isótopos + Rasgos: 694

Filas con isótopos Y rasgos:      258
Filas SOLO con isótopos:          417
Filas SOLO con rasgos:            19


In [9]:
# Igual que antes: revisamos en qué años están las filas "solo isótopos" (se espera que sean,
# sobre todo, años antiguos como 2006, pendientes de incorporar más adelante)
solo_iso = merge1[merge1["tiene_isotopos"] & ~merge1["tiene_rasgos"]]
print("Años de las filas 'solo isótopos' (sin rasgo emparejado):")
print(solo_iso["anio"].value_counts(dropna=False).sort_index())


Años de las filas 'solo isótopos' (sin rasgo emparejado):
anio
2005.0      4
2006.0    378
2007.0      2
2009.0      1
2011.0     26
2025.0      6
Name: count, dtype: int64


### Rasgos que todavía no tienen isótopo emparejado

Con la corrección del año, este número debería ser mucho más pequeño que antes (bajó de 127 a un
puñado de casos). Los que queden son casos genuinos que vale la pena revisar a mano: por ejemplo,
rasgos sin fecha de muestreo (`anio_rasgos` vacío) o árboles que realmente no tienen isótopo
registrado en ninguna base.


In [10]:
rasgos_sin_isotopos = merge1[merge1["tiene_rasgos"] & ~merge1["tiene_isotopos"]]
print("Filas de RASGOS sin isótopo emparejado:", len(rasgos_sin_isotopos))
rasgos_sin_isotopos[["id_arbol", "anio", "Site_rasgos", "PlotID", "genus_rasgos", "species_rasgos"]]


Filas de RASGOS sin isótopo emparejado: 19


,id_arbol,anio,Site_rasgos,PlotID,genus_rasgos,species_rasgos
5,3617.0,2025.0,Selva Viva,SEV_69,Browneopsis,NaN
10,3627.0,2025.0,Selva Viva,SEV_70,Guarea,kunthiana
15,3637.0,2025.0,Selva Viva,SEV_70,Pouteria,NaN
19,3647.0,2025.0,Selva Viva,SEV_70,Maclura,NaN
52,3715.0,2025.0,Sumaco,SUM_10,Miconia,aff glaucescens
147,8618.0,NaN,Galeras,GAL_50,Sapo28,NaN
190,8736.0,NaN,Galeras,GAL_28,Licania,macrocarpa
199,8767.0,NaN,Galeras,GAL_60,Eschweilera,NaN
201,8770.0,NaN,Galeras,GAL_60,Ocotea,NaN
208,8788.0,NaN,Galeras,GAL_50,Symplocos,cf spruceana


## Paso 5 — Preparar la dinámica forestal: DBH, área basal y crecimiento

### La corrección clave de la versión anterior

Antes, cuando un árbol tenía **filas duplicadas** en dinámica forestal, nos quedábamos solo con la
fila que tuviera `dbh_census3`, y si esa fila no lo tenía, el árbol quedaba marcado como "sin DBH" —
**aunque el DBH sí existiera en otra fila duplicada, en el censo 1 o 2**.

**Solución:** para cada árbol, combinamos **todas** sus filas duplicadas y armamos un valor de DBH
"final" con esta prioridad: **censo 3 → censo 2 → censo 1**. Ese valor final se pega a cada fila del
árbol en la tabla unificada, así que se repite en todas sus apariciones.

### Novedades de esta versión: área basal y crecimiento

- **Área basal**: es el área de la sección transversal del tronco, y se calcula directamente del DBH
  con la fórmula estándar de silvicultura:

  $$AB\ (m^2) = \frac{\pi}{4} \times \left(\frac{DBH\ (cm)}{100}\right)^2$$

  La calculamos para cada censo (1, 2 y 3) y también para el DBH "final".

- **Árboles muertos**: en dinámica forestal, cuando un árbol murió en un censo, su DBH casi siempre
  queda registrado como **0** (confirmamos que esto coincide casi perfectamente con la columna
  `dead_census*` = 1). Un DBH de 0 **no es una medida real**, así que antes de calcular cualquier área
  basal o crecimiento, "limpiamos" esos casos: si `dead_census* == 1`, tratamos ese DBH como vacío
  (`NaN`) en vez de como un valor real de 0. Esto evita que el crecimiento se vea artificialmente como
  "el árbol se encogió a cero".

- **Crecimiento (delta) de área basal**: para cada árbol, tomamos su **primera** y su **última**
  medición válida de DBH (pueden ser censo 1 y 3, o censo 1 y 2, o censo 2 y 3 — lo que esté
  disponible) y calculamos:
  - `delta_area_basal_m2` = área basal final − área basal inicial
  - `anios_entre_mediciones_dbh` = años transcurridos entre esas dos mediciones
  - `tasa_crecimiento_area_basal_m2_anio` = el delta dividido entre los años transcurridos (una tasa
    de crecimiento anual, útil para comparar árboles medidos en distintos intervalos de tiempo)


In [11]:
def primer_valido(serie):
    '''Devuelve el primer valor no nulo de una serie de datos; NaN si todos están vacíos.'''
    validos = serie.dropna()
    return validos.iloc[0] if len(validos) > 0 else np.nan


def area_basal_m2(dbh_cm):
    '''Calcula el área basal (m²) a partir de un DBH en centímetros.'''
    if pd.isna(dbh_cm):
        return np.nan
    return (np.pi / 4) * ((dbh_cm / 100) ** 2)


def combinar_filas_de_un_arbol(grupo):
    '''
    Recibe todas las filas de dinámica forestal de UN mismo árbol (puede ser 1 o varias filas
    duplicadas) y devuelve una sola fila combinada con:
      - los valores de DBH y área basal de cada censo, ya "limpios" (DBH de árboles muertos -> NaN),
      - un DBH y un área basal "final", priorizando el censo más reciente disponible,
      - el crecimiento (delta) de área basal entre la primera y la última medición válida,
      - una bandera de conflicto si hay valores de DBH distintos entre filas duplicadas.
    '''
    # Los flags de "muerto" a veces vienen como texto ('1', '0', 'c', etc.); los convertimos a número
    murio1 = pd.to_numeric(grupo["dead_census1"], errors="coerce")
    murio2 = pd.to_numeric(grupo["dead_census2"], errors="coerce")
    murio3 = pd.to_numeric(grupo["dead_census3"], errors="coerce")

    # Si el árbol murió en ese censo, su DBH (a menudo 0) no es una medida real: lo tratamos como NaN
    dbh1_limpio = grupo["dbh_census1"].where(murio1 != 1)
    dbh2_limpio = grupo["dbh_census2"].where(murio2 != 1)
    dbh3_limpio = grupo["dbh_census3"].where(murio3 != 1)

    dbh1 = primer_valido(dbh1_limpio); fecha1 = primer_valido(grupo["date_census1"])
    dbh2 = primer_valido(dbh2_limpio); fecha2 = primer_valido(grupo["date_census2"])
    dbh3 = primer_valido(dbh3_limpio); fecha3 = primer_valido(grupo["date_census3"])
    murio_census3 = primer_valido(grupo["dead_census3"])
    coment3 = primer_valido(grupo["comment_census3"])

    # DBH final: prioriza el censo más reciente que tenga dato disponible (ya limpio de "muertos")
    if pd.notna(dbh3):
        dbh_final, fecha_final, censo_usado = dbh3, fecha3, "censo_3"
    elif pd.notna(dbh2):
        dbh_final, fecha_final, censo_usado = dbh2, fecha2, "censo_2"
    elif pd.notna(dbh1):
        dbh_final, fecha_final, censo_usado = dbh1, fecha1, "censo_1"
    else:
        dbh_final, fecha_final, censo_usado = np.nan, np.nan, None

    # Área basal de cada censo y del DBH final
    ab1, ab2, ab3 = area_basal_m2(dbh1), area_basal_m2(dbh2), area_basal_m2(dbh3)
    ab_final = area_basal_m2(dbh_final)

    # Crecimiento: comparamos la primera y la última medición VÁLIDA de DBH (con año conocido)
    mediciones = []
    for dbh_v, fecha_v in [(dbh1, fecha1), (dbh2, fecha2), (dbh3, fecha3)]:
        if pd.notna(dbh_v) and pd.notna(fecha_v):
            anio_v = pd.to_datetime(fecha_v, errors="coerce").year
            if pd.notna(anio_v):
                mediciones.append((anio_v, area_basal_m2(dbh_v)))
    mediciones.sort(key=lambda par: par[0])

    if len(mediciones) >= 2:
        anio_ini, ab_ini = mediciones[0]
        anio_fin, ab_fin = mediciones[-1]
        anios_transcurridos = anio_fin - anio_ini
        delta_ab = ab_fin - ab_ini
        tasa = delta_ab / anios_transcurridos if anios_transcurridos > 0 else np.nan
    else:
        anio_ini = anio_fin = delta_ab = anios_transcurridos = tasa = np.nan

    # Conflicto: dentro del mismo censo, ¿las filas duplicadas dan valores DISTINTOS de DBH?
    conflicto = any(grupo[col].dropna().nunique() > 1 for col in ["dbh_census1", "dbh_census2", "dbh_census3"])

    return pd.Series({
        "dbh_census1_dinamica": dbh1, "fecha_census1_dinamica": fecha1,
        "dbh_census2_dinamica": dbh2, "fecha_census2_dinamica": fecha2,
        "dbh_census3_dinamica": dbh3, "fecha_census3_dinamica": fecha3,
        "muerto_census3_dinamica": murio_census3, "comentario_census3_dinamica": coment3,
        "dbh_final": dbh_final, "fecha_dbh_final": fecha_final, "censo_usado_dbh": censo_usado,
        "area_basal_census1_m2": ab1, "area_basal_census2_m2": ab2, "area_basal_census3_m2": ab3,
        "area_basal_final_m2": ab_final,
        "anio_dbh_inicial": anio_ini, "anio_dbh_final": anio_fin,
        "delta_area_basal_m2": delta_ab, "anios_entre_mediciones_dbh": anios_transcurridos,
        "tasa_crecimiento_area_basal_m2_anio": tasa,
        "conflicto_dbh": conflicto,
    })


# Solo trabajamos con filas que sí tienen id_arbol (sin ID no se puede unir)
dinamica_con_id = dinamica[dinamica["id_arbol"].notna()].copy()

# Agrupamos por árbol y combinamos sus filas (puede que sea 1 sola fila, o varias duplicadas)
dinamica_dedup = dinamica_con_id.groupby("id_arbol").apply(combinar_filas_de_un_arbol).reset_index()

print("Árboles únicos en dinámica forestal:", len(dinamica_dedup))
print("Árboles con conflicto de DBH entre filas duplicadas:", dinamica_dedup["conflicto_dbh"].sum())
print("Árboles con área basal final calculada:", dinamica_dedup["area_basal_final_m2"].notna().sum())
print("Árboles con crecimiento (delta) calculable (2+ mediciones válidas):",
      dinamica_dedup["delta_area_basal_m2"].notna().sum())
print()
print("¿De qué censo salió el DBH final?")
print(dinamica_dedup["censo_usado_dbh"].value_counts(dropna=False))


Árboles únicos en dinámica forestal: 1212
Árboles con conflicto de DBH entre filas duplicadas: 27
Árboles con área basal final calculada: 1210
Árboles con crecimiento (delta) calculable (2+ mediciones válidas): 660

¿De qué censo salió el DBH final?
censo_usado_dbh
censo_3    779
censo_1    292
censo_2    139
NaN          2
Name: count, dtype: int64


## Paso 6 — Unión 2: agregar el DBH al resultado anterior

Left join (unión izquierda) por `id_arbol`: partimos de `merge1` (isótopos + rasgos) y le agregamos
las columnas de DBH. Como el DBH ya viene combinado por árbol (un solo valor por `id_arbol`), se
repetirá automáticamente en **todas** las filas de ese árbol en la tabla final (sin importar en qué
año esté cada fila).


In [12]:
merge2 = pd.merge(
    merge1,
    dinamica_dedup,
    on="id_arbol",
    how="left",
)

merge2["tiene_dbh"] = merge2["dbh_final"].notna()

print("Filas totales después de agregar el DBH:", merge2.shape[0])
print("Filas con DBH disponible:    ", merge2["tiene_dbh"].sum())
print("Filas SIN DBH disponible:    ", (~merge2["tiene_dbh"]).sum())


Filas totales después de agregar el DBH: 694
Filas con DBH disponible:     668
Filas SIN DBH disponible:     26


In [13]:
# Verificación rápida: confirmamos que el DBH y el área basal SÍ se repiten en todas las filas
# de un mismo árbol
ejemplo_id = merge2.loc[merge2["id_arbol"].notna(), "id_arbol"].value_counts().index[0]
print(f"Ejemplo — árbol id_arbol={ejemplo_id}, todas sus filas en la tabla unificada:")
merge2[merge2["id_arbol"] == ejemplo_id][
    ["id_arbol", "anio", "tiene_isotopos", "tiene_rasgos", "dbh_final", "area_basal_final_m2", "censo_usado_dbh"]
]


Ejemplo — árbol id_arbol=3654.0, todas sus filas en la tabla unificada:


,id_arbol,anio,tiene_isotopos,tiene_rasgos,dbh_final,area_basal_final_m2,censo_usado_dbh
22,3654.0,2025.0,True,True,14.0,0.015394,censo_3
23,3654.0,2025.0,True,True,14.0,0.015394,censo_3
454,3654.0,2006.0,True,False,14.0,0.015394,censo_3
455,3654.0,2006.0,True,False,14.0,0.015394,censo_3


## Paso 7 — Cambio (delta) de isótopos entre años

Esta parte responde a una pregunta distinta a la de los pasos anteriores: no es "¿qué rasgo le
corresponde a este árbol?", sino "¿cómo cambió la composición isotópica de este árbol entre una
medición antigua y una reciente?". Por eso NO se calcula por fila (muestra), sino **por árbol**,
comparando su primer año de muestreo con su último año de muestreo.

**Ejemplo:** vimos antes que 158 árboles tienen isótopos medidos en más de un año (la mayoría en 2006
y 2025, o en 2011 y 2023). Para esos árboles calculamos:

- `d13C_anio_inicial` / `d13C_anio_final` y `delta_d13C` = valor final − valor inicial
- lo mismo para δ15N, `[C]%` y `[N]`
- `anio_isotopo_inicial`, `anio_isotopo_final` y `anios_entre_mediciones_isotopo`

Si un árbol solo tiene isótopos en un único año, su delta queda vacío (`NaN`) — no hay con qué
compararlo todavía.

**Nota:** si en una misma fecha hay más de una muestra (réplicas de laboratorio), promediamos esas
réplicas antes de calcular el delta, para no mezclar "variación entre réplicas" con "cambio real en
el tiempo".


In [14]:
# Variables de isótopos para las que calculamos el cambio en el tiempo
VARIABLES_ISOTOPO = {
    "δ13C (‰ v.s.V-PDB)": "d13C",
    "δ15N (‰ v.s. V-PDB)": "d15N",
    "[C]%": "C_pct",
    "[N]": "N_pct",
}


def combinar_isotopos_de_un_arbol(grupo):
    '''
    Recibe TODAS las filas de isótopos de un mismo árbol (de cualquier año) y calcula, para cada
    variable de interés, el valor en su primer año de muestreo, en su último año, y la diferencia
    entre ambos (delta). Si el árbol solo tiene un año de datos, el delta queda vacío.
    '''
    salida = {}
    anios_validos = []

    for columna_original, alias in VARIABLES_ISOTOPO.items():
        # Promediamos por año, por si hay más de una réplica de laboratorio el mismo año
        promedio_por_anio = grupo.groupby("anio_isotopo")[columna_original].mean().dropna()

        if len(promedio_por_anio) >= 2:
            anio_ini = promedio_por_anio.index.min()
            anio_fin = promedio_por_anio.index.max()
            valor_ini = promedio_por_anio.loc[anio_ini]
            valor_fin = promedio_por_anio.loc[anio_fin]

            salida[f"{alias}_anio_inicial"] = valor_ini
            salida[f"{alias}_anio_final"] = valor_fin
            salida[f"delta_{alias}"] = valor_fin - valor_ini
            anios_validos.append((anio_ini, anio_fin))
        else:
            salida[f"{alias}_anio_inicial"] = np.nan
            salida[f"{alias}_anio_final"] = np.nan
            salida[f"delta_{alias}"] = np.nan

    if anios_validos:
        salida["anio_isotopo_inicial"] = min(par[0] for par in anios_validos)
        salida["anio_isotopo_final"] = max(par[1] for par in anios_validos)
        salida["anios_entre_mediciones_isotopo"] = salida["anio_isotopo_final"] - salida["anio_isotopo_inicial"]
    else:
        salida["anio_isotopo_inicial"] = np.nan
        salida["anio_isotopo_final"] = np.nan
        salida["anios_entre_mediciones_isotopo"] = np.nan

    return pd.Series(salida)


# Usamos la base de isótopos COMPLETA (todos los años), no solo la reciente, porque el delta
# necesita comparar años lejanos (ej. 2006 vs 2025)
iso_con_id = iso[iso["id_arbol"].notna()].copy()
deltas_isotopos = iso_con_id.groupby("id_arbol").apply(combinar_isotopos_de_un_arbol).reset_index()

print("Árboles con isótopos:", len(deltas_isotopos))
print("Árboles con delta de isótopo calculable (2+ años de muestreo):",
      deltas_isotopos["anios_entre_mediciones_isotopo"].notna().sum())
print()
deltas_isotopos[deltas_isotopos["anios_entre_mediciones_isotopo"].notna()][
    ["id_arbol", "anio_isotopo_inicial", "anio_isotopo_final", "delta_d13C", "delta_d15N"]
].head(10)


Árboles con isótopos: 409
Árboles con delta de isótopo calculable (2+ años de muestreo): 158



,id_arbol,anio_isotopo_inicial,anio_isotopo_final,delta_d13C,delta_d15N
95,3605.0,2006.0,2025.0,0.181575,NaN
96,3606.0,2006.0,2025.0,-0.279980,NaN
105,3622.0,2006.0,2025.0,1.111591,NaN
106,3623.0,2006.0,2025.0,0.525583,NaN
107,3624.0,2006.0,2025.0,-0.750000,NaN
113,3631.0,2006.0,2025.0,-3.130000,NaN
115,3634.0,2006.0,2025.0,-3.556214,NaN
116,3635.0,2006.0,2025.0,-0.320000,NaN
121,3642.0,2006.0,2025.0,-1.233686,NaN
122,3643.0,2006.0,2025.0,-1.021955,NaN


### Agregar los deltas de isótopos y de área basal a la tabla unificada

Igual que con el DBH, unimos estos resultados por `id_arbol` (left join), así que cada árbol lleva su
delta "pegado" en todas las filas donde aparece, sin importar el año de esa fila en particular.


In [15]:
merge2 = pd.merge(merge2, deltas_isotopos, on="id_arbol", how="left")

print("Columnas totales después de agregar los deltas de isótopos:", merge2.shape[1])


Columnas totales después de agregar los deltas de isótopos: 110


## Paso 8 — Ordenar las columnas para que la tabla final sea legible

In [16]:
orden_deseado = [
    # Identificadores y banderas de control
    "id_arbol", "anio", "anio_isotopo", "tiene_isotopos", "tiene_rasgos", "tiene_dbh",
    "conflicto_dbh",
    # Ubicación
    "Site_iso", "Site_rasgos", "Plot_iso", "Plot_rasgos", "PlotID", "Subplot",
    "Elevation_m", "altitude_m",
    # Taxonomía
    "family_iso", "genus_iso", "species_iso",
    "family_rasgos", "genus_rasgos", "species_rasgos", "genus_species",
    # IDs originales (trazabilidad hacia las bases de origen)
    "treeID", "old_treeID", "new_TreeID", "new_tree_ID_2025",
    # Isótopos
    "Sample_ID", "Year", "Type", "Type 2",
    "[C]%", "δ13C (‰ v.s.V-PDB)", "[C_b]%", "bulk_δ13C (‰ v.s.V-PDB)",
    "[N]", "δ15N (‰ v.s. V-PDB)", "description",
    "JH herbarium collections", "numeroCampo", "leaf_sample_ID_2011", "Plot_from_ID",
    # Cambio (delta) de isótopos en el tiempo
    "anio_isotopo_inicial", "anio_isotopo_final", "anios_entre_mediciones_isotopo",
    "d13C_anio_inicial", "d13C_anio_final", "delta_d13C",
    "d15N_anio_inicial", "d15N_anio_final", "delta_d15N",
    "C_pct_anio_inicial", "C_pct_anio_final", "delta_C_pct",
    "N_pct_anio_inicial", "N_pct_anio_final", "delta_N_pct",
    # Rasgos funcionales
    "dataset_source", "sampling_date", "n_leaves",
    "leaf_fresh_weight_g", "leaf_dry_weight_g", "leaf_area_cm2",
    "SLA_cm2_g", "LDMC_mg_g", "mean_leaf_thickness_mm",
    "wood_density_g_cm3", "WSG", "stem_water_content_pct", "force_to_punch_kN_m",
    "C_mg_g", "N_mg_g", "P_mg_g", "Al_mg_g", "Ca_mg_g", "Fe_mg_g", "K_mg_g", "Mg_mg_g", "Na_mg_g", "S_mg_g",
    # Dinámica forestal: DBH y área basal
    "dbh_final", "fecha_dbh_final", "censo_usado_dbh", "area_basal_final_m2",
    "dbh_census1_dinamica", "fecha_census1_dinamica", "area_basal_census1_m2",
    "dbh_census2_dinamica", "fecha_census2_dinamica", "area_basal_census2_m2",
    "dbh_census3_dinamica", "fecha_census3_dinamica", "area_basal_census3_m2",
    "muerto_census3_dinamica", "comentario_census3_dinamica",
    # Cambio (delta) de área basal en el tiempo
    "anio_dbh_inicial", "anio_dbh_final", "anios_entre_mediciones_dbh",
    "delta_area_basal_m2", "tasa_crecimiento_area_basal_m2_anio",
    # Calidad y trazabilidad
    "QC_flag_iso", "QC_flag_rasgos", "comment_iso", "comment_rasgos",
    "source_file_iso", "source_file_rasgos",
]

orden_existente = [c for c in orden_deseado if c in merge2.columns]
columnas_restantes = [c for c in merge2.columns if c not in orden_existente]

tabla_final = merge2[orden_existente + columnas_restantes]

print("Total de columnas en la tabla final:", tabla_final.shape[1])
print("Columnas que quedaron sin ubicar manualmente (van al final):", columnas_restantes)


Total de columnas en la tabla final: 110
Columnas que quedaron sin ubicar manualmente (van al final): ['id_nuevo_iso', 'id_viejo_iso', 'id_nuevo_rasgos', 'id_viejo_rasgos', 'anio_rasgos']


## Paso 9 — Preparar las hojas de revisión manual

In [17]:
# 1) Filas con isótopos y/o rasgos que NO tienen DBH -> para revisión manual cuidadosa
faltan_dbh = tabla_final[
    (tabla_final["tiene_rasgos"] | tabla_final["tiene_isotopos"]) & (~tabla_final["tiene_dbh"])
].copy()

# 2) Filas de rasgos que no encontraron isótopo
rasgos_sin_isotopos_final = tabla_final[
    tabla_final["tiene_rasgos"] & (~tabla_final["tiene_isotopos"])
].copy()

# 3) Árboles con conflicto de DBH entre filas duplicadas de dinámica forestal
conflictos_dbh = tabla_final[tabla_final["conflicto_dbh"] == True].copy()

print("Filas para la hoja 'Faltan_DBH':          ", len(faltan_dbh))
print("Filas para la hoja 'Rasgos_sin_isotopos':  ", len(rasgos_sin_isotopos_final))
print("Filas para la hoja 'Conflictos_DBH':       ", len(conflictos_dbh))


Filas para la hoja 'Faltan_DBH':           26
Filas para la hoja 'Rasgos_sin_isotopos':   19
Filas para la hoja 'Conflictos_DBH':        21


### Hojas de resumen para los cambios en el tiempo (una fila por árbol)

`tabla_final` tiene una fila por **muestra** (puede haber varias filas para el mismo árbol, una por
año). Para revisar los deltas es más cómodo tener **una sola fila por árbol**, con su información de
ubicación y taxonomía. Armamos esas dos hojas de resumen aquí.


In [18]:
# Tomamos, para cada árbol, una fila de referencia con su ubicación y taxonomía
# (usamos la de isótopos si existe; si no, la de rasgos)
columnas_contexto = ["id_arbol", "Site_iso", "Site_rasgos", "genus_iso", "species_iso",
                     "genus_rasgos", "species_rasgos"]
contexto_por_arbol = (
    tabla_final[columnas_contexto]
    .dropna(subset=["id_arbol"])
    .drop_duplicates(subset="id_arbol", keep="first")
)
contexto_por_arbol["Site"] = contexto_por_arbol["Site_iso"].combine_first(contexto_por_arbol["Site_rasgos"])
contexto_por_arbol["genus"] = contexto_por_arbol["genus_iso"].combine_first(contexto_por_arbol["genus_rasgos"])
contexto_por_arbol["species"] = contexto_por_arbol["species_iso"].combine_first(contexto_por_arbol["species_rasgos"])
contexto_por_arbol = contexto_por_arbol[["id_arbol", "Site", "genus", "species"]]

# Hoja 1: crecimiento de área basal (una fila por árbol con delta calculable)
crecimiento_area_basal = pd.merge(contexto_por_arbol, dinamica_dedup, on="id_arbol", how="inner")
crecimiento_area_basal = crecimiento_area_basal[crecimiento_area_basal["delta_area_basal_m2"].notna()].copy()
crecimiento_area_basal = crecimiento_area_basal.sort_values("tasa_crecimiento_area_basal_m2_anio", ascending=False)

# Hoja 2: cambio de isótopos (una fila por árbol con delta calculable)
cambios_isotopos = pd.merge(contexto_por_arbol, deltas_isotopos, on="id_arbol", how="inner")
cambios_isotopos = cambios_isotopos[cambios_isotopos["anios_entre_mediciones_isotopo"].notna()].copy()

print("Árboles en la hoja 'Crecimiento_area_basal':", len(crecimiento_area_basal))
print("Árboles en la hoja 'Cambios_isotopos':      ", len(cambios_isotopos))


Árboles en la hoja 'Crecimiento_area_basal': 261
Árboles en la hoja 'Cambios_isotopos':       158


In [19]:
resumen = pd.DataFrame({
    "Indicador": [
        "Filas en isótopos (original)",
        "Filas en rasgos (original)",
        "Filas de rasgos excluidas por ser de la base 'SF'",
        "Filas en dinámica forestal (original)",
        "Filas en tabla unificada final",
        "Filas con isótopos Y rasgos emparejados",
        "Filas de rasgos SIN isótopo emparejado",
        "Filas con isótopos y/o rasgos que NO tienen DBH",
        "Árboles con conflicto de DBH entre filas duplicadas",
        "Árboles con crecimiento de área basal calculable",
        "Árboles con cambio (delta) de isótopos calculable",
    ],
    "Valor": [
        len(iso),
        n_antes,
        n_antes - n_despues,
        len(dinamica),
        len(tabla_final),
        int((tabla_final["tiene_isotopos"] & tabla_final["tiene_rasgos"]).sum()),
        len(rasgos_sin_isotopos_final),
        len(faltan_dbh),
        int(tabla_final["conflicto_dbh"].sum()),
        len(crecimiento_area_basal),
        len(cambios_isotopos),
    ],
})
resumen


,Indicador,Valor
0,Filas en isótopos (original),669
1,Filas en rasgos (original),313
2,Filas de rasgos excluidas por ser de la base 'SF',51
3,Filas en dinámica forestal (original),1336
4,Filas en tabla unificada final,694
5,Filas con isótopos Y rasgos emparejados,258
6,Filas de rasgos SIN isótopo emparejado,19
7,Filas con isótopos y/o rasgos que NO tienen DBH,26
8,Árboles con conflicto de DBH entre filas dupli...,21
9,Árboles con crecimiento de área basal calculable,261


## Paso 10 — Exportar el resultado a un único archivo Excel

- `Resumen`: indicadores generales.
- `Unificado_completo`: toda la información unida (isótopos + rasgos + DBH + área basal + deltas).
- `Faltan_DBH`: para revisar manualmente a quiénes les falta el DBH.
- `Rasgos_sin_isotopos`: rasgos que no encontraron isótopo pareja.
- `Conflictos_DBH`: árboles cuyas filas duplicadas en dinámica forestal tienen valores de DBH distintos.
- `Crecimiento_area_basal`: una fila por árbol, con su delta de área basal (los que sí se pudieron
  calcular), ordenados de mayor a menor tasa de crecimiento.
- `Cambios_isotopos`: una fila por árbol, con su delta de δ13C, δ15N, [C]% y [N] (los que sí se
  pudieron calcular).


In [20]:
import os

os.makedirs("output", exist_ok=True)
ruta_salida = "output/Base_unificada_isotopos_rasgos_dbh.xlsx"

with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="Resumen", index=False)
    tabla_final.to_excel(writer, sheet_name="Unificado_completo", index=False)
    faltan_dbh.to_excel(writer, sheet_name="Faltan_DBH", index=False)
    rasgos_sin_isotopos_final.to_excel(writer, sheet_name="Rasgos_sin_isotopos", index=False)
    conflictos_dbh.to_excel(writer, sheet_name="Conflictos_DBH", index=False)
    crecimiento_area_basal.to_excel(writer, sheet_name="Crecimiento_area_basal", index=False)
    cambios_isotopos.to_excel(writer, sheet_name="Cambios_isotopos", index=False)

print("Archivo guardado en:", ruta_salida)


Archivo guardado en: output/Base_unificada_isotopos_rasgos_dbh.xlsx


## Notas finales / próximos pasos

- Cuando tengas lista la base de **isótopos del 2006**, repite el Paso 4 (la unión reciente sigue
  igual; solo se ampliaría el bloque "histórico" con datos de rasgos si llegaran a existir para esos
  años).
- La hoja **`Faltan_DBH`** es el punto de partida sugerido para tu revisión manual.
- La hoja **`Conflictos_DBH`** merece revisión aparte: son árboles donde la dinámica forestal tiene
  filas duplicadas con DBH distintos en el mismo censo — probablemente un error de digitación o de
  identificación de árbol que vale la pena aclarar en la fuente original.
- Si en algún caso el DBH "final" debería salir de un censo distinto al que el código escogió
  automáticamente, la columna `censo_usado_dbh` indica de cuál salió, y `dbh_census1_dinamica`,
  `dbh_census2_dinamica` y `dbh_census3_dinamica` quedan disponibles para comparar y corregir a mano.
- **Sobre los deltas (área basal e isótopos):** quedaron calculados en este mismo notebook y también
  como columnas dentro de `Unificado_completo` (se repiten en todas las filas del árbol
  correspondiente), además de sus propias hojas resumen. Si más adelante quieres profundizar en el
  análisis (por ejemplo, relacionar el crecimiento con el cambio isotópico, hacer gráficos, o separar
  por sitio/especie), lo natural es partir de las hojas `Crecimiento_area_basal` y `Cambios_isotopos`
  —ya tienen una fila por árbol y son la base ideal para ese análisis, sea en este mismo notebook (un
  nuevo bloque de "Paso 11 en adelante") o en un notebook aparte que las lea directamente desde este
  Excel.
